In [ ]:
pip install matplotlib-venn

In [ ]:
# https://pypi.python.org/pypi/libarchive
!apt-get -qq install -y libarchive-dev && pip install -U libarchive
import libarchive

In [ ]:
!pip install cartopy

In [ ]:
pip install pydot

In [ ]:

!pip install yfinance

In [ ]:

!pip install scikit-learn

In [ ]:
pip install StandardScaler

In [ ]:

import yfinance as yf
import pandas as pd

# Define the currency pairs of interest
pairs = ['EURUSD=X', 'EURJPY=X', 'EURAUD=X', 'GBPUSD=X', 'USDCAD=X', 'USDCHF=X']

# Set the start and end dates for the data
start_date = '2010-01-01'
end_date = '2022-02-28'

# Collect the data for each currency pair
dfs = []
for pair in pairs:
    data = yf.download(pair, start=start_date, end=end_date)
    data['Pair'] = pair
    dfs.append(data)

# Concatenate the data into a single dataframe
df = pd.concat(dfs)

# Save the data to a CSV file
df.to_csv('currency_data.csv')

In [ ]:

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from keras.models import Sequential
from keras.layers import Dense

# Load the data
data = pd.read_csv('currency_data.csv')

# Scale the data
scaler = StandardScaler()
scaled_data = scaler.fit_transform(data.drop(['Label'], axis=1))


# Define the input shape
input_shape = (scaled_data.shape[1],)


# Define the model architecture
model = Sequential()
model.add(Dense(16, input_shape=input_shape, activation='relu'))
model.add(Dense(8, activation='relu'))
model.add(Dense(1, activation='sigmoid'))


# Compile the model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train the model
model.fit(scaled_data, data['Label'], epochs=10, batch_size=32)

# Define the input shape
input_shape = (scaled_data.shape[1],)

# Define the model
input_ = tf.keras.layers.Input(shape=input_shape)
dense1 = tf.keras.layers.Dense(32, activation='relu')(input_)
dense2 = tf.keras.layers.Dense(16, activation='relu')(dense1)
output = tf.keras.layers.Dense(1, activation='sigmoid')(dense2)
model = tf.keras.models.Model(inputs=input_, outputs=output)

# Compile the model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train the model
features = scaled_data
labels = data['Label']
model.fit(features, labels, epochs=100, batch_size=32)

# Save the model
model.save('trading_system_model.h5')

# ส่วนใหม่

In [ ]:

import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model

# Load the historical data
df = pd.read_csv('currency_data.csv')

# Load the AI model
model = load_model('trading_system_model.h5')

# Initialize trading parameters
balance = 100000.0
position_size = 10000.0

# Simulate trades
predictions = []
for i in range(len(df)):
    # Get the current data point
    data_point = df.iloc[i, 1:5]
    
    # Preprocess the data point (in the same way as for training)
    data_point = (data_point - df.iloc[:, 1:5].mean()) / df.iloc[:, 1:5].std(ddof=1)
    
    # Make a prediction using the AI model
    data_point = np.array(data_point).reshape(1, -1)
    prediction = model.predict(tf.convert_to_tensor(data_point, dtype=tf.float32)).flatten()
    predictions.append(prediction)

# Evaluate performance
pnl = balance - 100000.0
win_rate = sum(np.diff(np.sign(np.diff(df['Close']))) > 0) / len(df)
drawdown = min([max(np.maximum.accumulate(df['Close']) - df['Close']) / np.maximum.accumulate(df['Close'])])
sharpe_ratio = np.sqrt(len(df)) * np.mean(np.diff(df['Close'])) / np.std(np.diff(df['Close']))
print(f'PnL: {pnl:.2f}, Win rate: {win_rate:.2f}, Drawdown: {drawdown:.2f}, Sharpe ratio: {sharpe_ratio:.2f}')